# STGAN su ERA5: cosa vede solo il modello, e con quale anticipo

Due domande, con grafici generali (nessun caso singolo):
- **A. Cosa segnala solo il modello?** Quanto delle sue segnalazioni i rilevatori senza modello non vedono, dove
  cadono quei valori per gli altri rilevatori, e se corrispondono a tempo davvero insolito.
- **B. Con quale anticipo?** Nel tempo, nella stessa cella; e nello spazio, nelle celle intorno, prima che un
  episodio dei rilevatori senza modello cominci.

I rilevatori, tutti dopo la climatologia (standardizzati per cella, mese e ora):

| Rilevatore | Cosa misura | Modello? |
|---|---|---|
| **modello** | errore di ricostruzione del generatore, media dei campioni MC Dropout | sì |
| **distanza dal normale** | quanto il valore si allontana dal normale stagionale di cella, giorno e ora | no |
| **variazione in 3 h** | quanto il valore è cambiato rispetto all'istante prima | no |
| **normale + variazione** | il più alto dei due precedenti: il riferimento senza modello | no |

Un valore (cella × istante) è **segnalato** da un rilevatore se è nel suo 5% più alto oppure, nella seconda
lettura, nel suo 1% più alto (solo le segnalazioni più forti). Ogni rilevatore segnala la stessa quota di valori,
quindi il confronto è alla pari.

**Da dove vengono i numeri.** Le sezioni A1 e A2 sono calcolate qui dagli score della run, su un istante ogni
`STRIDE`. Le sezioni A3, B1 e B2 leggono i log e le tabelle dei controlli lanciati a parte sul server, su tutti gli
istanti, nella cartella `ANALYSIS`; ogni sezione dice da quale script vengono. Se un file manca, la sezione lo dice
e si passa alla successiva.

**Avvertenze.** Run `_ep3`, un solo seed. Climatologia e soglie sono stimate sugli anni di test: è una prova
delle figure. I numeri per la tesi verranno dalla run scelta dallo sweep, con climatologia e soglie stimate sulla
validazione; per rifare il notebook basta cambiare `RUN` e `ANALYSIS`.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

ROOT = Path(os.environ['PHYSIQ_ROOT']) if os.environ.get('PHYSIQ_ROOT') else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla cartella del repository physiq_pv')
OUT = ROOT / 'outputs'
RUN = OUT / 'stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3'
ANALYSIS = OUT / 'analisi_solo_modello'  # log e tabelle dei controlli lanciati a parte
SCORES = {  # stesso periodo e stesse celle, tutti dopo la climatologia
    'modello': Path(f'{RUN}_gen_clim'),
    'distanza dal normale': Path(f'{RUN}_no_model_bal_clim'),
    'variazione in 3 h': Path(f'{RUN}_previous_bal_clim'),
}
MODEL, NORMAL, CHANGE = SCORES
SIMPLE = 'normale + variazione'
DETECTORS = [MODEL, NORMAL, CHANGE, SIMPLE]
FOLDER = {'gen_clim': MODEL, 'simple_max': SIMPLE}  # nomi delle cartelle che compaiono nei log
LEVELS = (5, 1)     # segnalato = nel 5% / nell'1% più alto di ciascun rilevatore
STRIDE = 29         # un istante ogni 29 (dispari: si leggono tutte le ore del giorno)
GROUPS = ['solo modello', 'entrambi', 'solo semplici', 'nessuno']
GROUP_COLORS = ['tab:red', 'tab:purple', 'tab:blue', 'tab:grey']
LEVEL_COLORS = {5: 'tab:orange', 1: 'tab:red'}
SAVE_FIGURES = False
FIGURE_DIR = OUT / 'figure_solo_modello'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

def show(figure, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight', dpi=150)
    plt.show()

def manifest(directory):
    return json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))

def open_scores(directory):
    data = manifest(directory)
    return np.load(directory / data.get('anomaly_mean_file', data.get('scores_file', 'scores/anomaly_mean.npy')),
                   mmap_mode='r')

def find(name):
    path = ANALYSIS / name
    if not path.is_file():
        print(f'File non trovato: {path}')
        return None
    return path

def lines_of(name):
    path = find(name)
    return None if path is None else path.read_text(encoding='utf-8', errors='replace').splitlines()

---

# A. Cosa segnala solo il modello

Lettura degli score: un istante ogni `STRIDE`, tutte le celle. "Normale + variazione" è il più alto dei due score
semplici, come nella cartella `_simple_max` dei controlli sul server.

In [ ]:
times = pd.to_datetime(np.load(SCORES[MODEL] / 'test_timestamps.npy'))
rows_read = np.arange(0, len(times), STRIDE)
value = {}
for name, directory in SCORES.items():
    if not pd.to_datetime(np.load(directory / 'test_timestamps.npy')).equals(times):
        raise ValueError(f'{name}: non è sugli stessi istanti del modello')
    scores = open_scores(directory)
    value[name] = np.asarray(scores[rows_read], dtype=np.float32)
    del scores
finite = np.logical_and.reduce([np.isfinite(values) for values in value.values()])
value[SIMPLE] = np.fmax(value[NORMAL], value[CHANGE])
limit = {level: {name: float(np.percentile(value[name][finite], 100 - level)) for name in DETECTORS}
         for level in LEVELS}
flag = {level: {name: (value[name] > limit[level][name]) & finite for name in DETECTORS} for level in LEVELS}
print(f'{len(rows_read)} istanti letti, dal {times[rows_read[0]].date()} al {times[rows_read[-1]].date()}; '
      f'{finite.shape[1]} celle')
display(pd.DataFrame(limit).rename(columns=lambda level: f'soglia {level}%').round(3))

## A1. Quante segnalazioni del modello gli altri non vedono

Per ogni rilevatore senza modello: delle segnalazioni del modello, la quota che quel rilevatore **non** segnala
allo stesso livello. La barra di "normale + variazione" è il gruppo **solo modello** delle sezioni A2 e A3.

In [ ]:
OTHERS = [NORMAL, CHANGE, SIMPLE]
alone_share = pd.DataFrame({f'{level}%': {other: (flag[level][MODEL] & ~flag[level][other]).sum()
                                                 / flag[level][MODEL].sum() for other in OTHERS}
                            for level in LEVELS})
figure, axis = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
positions = np.arange(len(OTHERS))
height = 0.8 / len(LEVELS)
for index, level in enumerate(LEVELS):
    shares = 100 * alone_share[f'{level}%'].to_numpy()
    bars = axis.barh(positions + index * height, shares, height=height, color=LEVEL_COLORS[level],
                     label=f'segnalati al {level}%')
    axis.bar_label(bars, fmt='%.0f%%', fontsize=8, padding=2)
axis.set_yticks(positions + height * (len(LEVELS) - 1) / 2, [f'non visti da {other}' for other in OTHERS])
axis.set(xlim=(0, 100), xlabel='% delle segnalazioni del modello',
         title="Segnalazioni del modello che l'altro rilevatore non vede")
axis.invert_yaxis(); axis.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
show(figure, 'a1_non_visti')
display(alone_share.style.format('{:.1%}'))

## A2. Per gli altri rilevatori, quanto sono insoliti i valori "solo modello"?

I valori segnalati dal modello e non da "normale + variazione": per ciascuno, il **percentile** che gli danno i
rilevatori senza modello (100 = il più insolito di tutti). Due letture opposte:
- valori **appena sotto la soglia** degli altri (fra il 90° e il 95° percentile al 5%, fra il 98° e il 99° all'1%):
  il modello segnala le stesse cose, solo con una soglia diversa;
- valori **ordinari** per gli altri (sotto il 50° percentile): il modello vede qualcosa che la distanza dal
  normale e la variazione non vedono.

La linea tratteggiata è la distribuzione di tutti i valori (uniforme per costruzione); quella punteggiata è la
soglia di "normale + variazione".

In [ ]:
reference = {name: np.sort(value[name][finite]) for name in OTHERS}

def percentile_of(name, mask):
    return 100 * np.searchsorted(reference[name], value[name][mask], side='right') / len(reference[name])

bins = np.arange(0, 100.01, 2)
figure, axes = plt.subplots(len(LEVELS), len(OTHERS), figsize=(5 * len(OTHERS), 3.4 * len(LEVELS)),
                            constrained_layout=True, squeeze=False, sharey='row')
summary = {}
for row, level in enumerate(LEVELS):
    alone = flag[level][MODEL] & ~flag[level][SIMPLE]
    for column, other in enumerate(OTHERS):
        percentiles = percentile_of(other, alone)
        axis = axes[row, column]
        axis.hist(percentiles, bins=bins, weights=np.full(len(percentiles), 100 / max(len(percentiles), 1)),
                  color=LEVEL_COLORS[level], alpha=0.85)
        axis.axhline(100 * (bins[1] - bins[0]) / 100, color='black', linestyle='--', linewidth=1)
        axis.axvline(100 - level, color='black', linestyle=':', linewidth=1)
        axis.set(xlim=(0, 100), xlabel=f'percentile in {other}',
                 ylabel=f'% dei valori solo modello ({level}%)' if column == 0 else '')
        if row == 0:
            axis.set_title(f'giudizio di: {other}')
        summary[f'{level}%', other] = {
            'valori': len(percentiles),
            'percentile mediano': np.median(percentiles),
            f'appena sotto la soglia': np.mean((percentiles >= 100 - 2 * level) & (percentiles <= 100 - level)),
            'sopra la sua soglia': np.mean(percentiles > 100 - level),
            'ordinari (sotto il 50°)': np.mean(percentiles < 50)}
show(figure, 'a2_percentili_solo_modello')
summary = pd.DataFrame(summary).T
display(summary.style.format({'valori': '{:.0f}', 'percentile mediano': '{:.0f}', 'appena sotto la soglia': '{:.1%}',
                              'sopra la sua soglia': '{:.1%}', 'ordinari (sotto il 50°)': '{:.1%}'}))
print("'appena sotto la soglia': fra il 90° e il 95° percentile al 5%, fra il 98° e il 99° all'1%. "
      "'sopra la sua soglia': per i due score semplici presi da soli, che hanno una soglia più bassa del loro massimo.")

## A3. Quello che vede solo il modello è tempo davvero insolito?

I valori sono divisi in quattro gruppi rispetto a "normale + variazione", allo stesso livello:

| Gruppo | Modello | Normale + variazione |
|---|---|---|
| **solo modello** | ✔ | ✘ |
| **entrambi** | ✔ | ✔ |
| **solo semplici** | ✘ | ✔ |
| **nessuno** | ✘ | ✘ |

Per ogni gruppo, quante volte una grandezza **che nessuno score usa** è insolita, diviso la stessa quota su tutti i
valori: 1 = come il caso, 2 = due volte più spesso. "Insolito" vuol dire nel 5% più estremo per cella e mese (e
ora, per ERA5), a entrambi i livelli di segnalazione.
- **CAPE** (energia disponibile per i temporali) e **raffica** di vento a 10 m: variabili ERA5 scaricate a parte,
  su tutto il dominio. Script `check_era5_score_against_unseen.py`, log `non_viste_<livello>.log`.
- **Stazioni E-OBS** (solo terraferma europea, per giorno): pioggia forte, pressione bassa, calo di pressione in un
  giorno, radiazione bassa, caldo, freddo. Script `check_era5_score_against_eobs.py`, colonna "same day", log
  `eobs_<indicatore>_<livello>.log`.

In [ ]:
UNSEEN_ROWS = {'score alone, none of the others': 'solo modello', 'score and at least one other': 'entrambi',
               'others, not the score': 'solo semplici', 'flagged by none': 'nessuno', 'all values': 'tutti'}
UNSEEN_NAMES = {'cape': 'CAPE alta', 'i10fg': 'raffica alta'}
EOBS_ROWS = {'score alone': 'solo modello', 'score and another': 'entrambi', 'others only': 'solo semplici',
             'none': 'nessuno', 'all cell-days': 'tutti'}
EOBS = {'pioggia': 'pioggia forte', 'pressione_bassa': 'pressione bassa', 'calo_pressione': 'calo di pressione',
        'radiazione_bassa': 'radiazione bassa', 'caldo': 'caldo', 'freddo': 'freddo'}

def read_unseen(level):
    # First table of the log: % of the values of each group where each unseen variable is unusual.
    lines = lines_of(f'non_viste_{level}.log')
    if lines is None:
        return {}
    header, table = None, {}
    for line in lines:
        tokens = line.split()
        if header is None:
            if tokens[:1] == ['values']:
                header = tokens[1:]
            continue
        if line.startswith('% of the values where'):
            break
        parts = re.split(r'\s{2,}', line.strip())
        if parts[0] in UNSEEN_ROWS and len(parts) == len(header) + 2:
            for variable, number in zip(header, parts[2:]):
                table.setdefault(UNSEEN_NAMES.get(variable, variable), {})[UNSEEN_ROWS[parts[0]]] = float(number)
    return table

def read_eobs(key, level):
    # First table of the log: % of the cell-days of each group with the unusual measure, same day.
    lines = lines_of(f'eobs_{key}_{level}.log')
    if lines is None:
        return None
    row = {}
    for line in lines:
        if line.startswith(('same day:', 'Ratios')):
            break
        label, numbers = line[:28].strip(), line[28:].split()
        if label in EOBS_ROWS and len(numbers) == 4:
            row[EOBS_ROWS[label]] = float(numbers[1])
    return row or None

shares = {}
for level in LEVELS:
    table = read_unseen(level)
    for key, name in EOBS.items():
        row = read_eobs(key, level)
        if row is not None:
            table[name] = row
    if table:
        shares[level] = pd.DataFrame(table).T.reindex(columns=GROUPS + ['tutti'])
if shares:
    figure, axes = plt.subplots(len(shares), 1, figsize=(12, 3.8 * len(shares)), constrained_layout=True,
                                squeeze=False)
    for axis, (level, table) in zip(axes[:, 0], shares.items()):
        ratios = table[GROUPS].div(table['tutti'], axis=0)
        ratios.plot.bar(ax=axis, color=GROUP_COLORS, width=0.8, rot=0)
        axis.axhline(1, color='black', linestyle='--', linewidth=1)
        axis.set(ylabel='volte il caso', xlabel='',
                 title=f'Segnalati al {level}%: quante volte più spesso del caso la grandezza è insolita')
        axis.legend(fontsize=8, ncol=4, loc='upper right')
    show(figure, 'a3_tempo_insolito')
    for level, table in shares.items():
        print(f'Segnalati al {level}%: % dei valori del gruppo con la grandezza insolita ("tutti" = il caso)')
        display(table.round(1))

---

# B. Con quale anticipo

## B1. Nel tempo, nella stessa cella

Un **episodio** di un rilevatore comincia in una cella quando la segnala dopo almeno 24 ore senza segnalarla.
All'inizio di ogni episodio di "normale + variazione", il modello segnalava **già** la stessa cella, e da quante
ore? E al contrario: all'inizio di ogni episodio del modello, "normale + variazione" segnalava già?

Le curve dicono la quota degli inizi in cui l'altro era già acceso da **almeno** h ore. Le tratteggiate sono il
caso: la stessa misura in celle e istanti dove il rilevatore della curva era quieto da altrettanto e non comincia
nulla. Script `check_era5_lead.py`, log `anticipo_<livello>.log`.

In [ ]:
LEAD_COLUMNS = ['starts', 'already', '3h', '6h', '9-12h', '15-24h', '>24h', 'lead med h', 'gap', 'together',
                'later', 'lag med h', 'none']
LEAD_BANDS = ['3h', '6h', '9-12h', '15-24h', '>24h']
AT_LEAST = [3, 6, 9, 15, 27]  # ore: "già acceso da almeno" (dalle fasce dello script)

def read_lead(level):
    lines = lines_of(f'anticipo_{level}.log')
    if lines is None:
        return None
    rows = {}
    for line in lines:
        if line.startswith(('starts of', '  chance')):
            values = line[34:].split()
            if len(values) == len(LEAD_COLUMNS):
                rows[line[:34].strip()] = dict(zip(LEAD_COLUMNS, map(float, values)))
    return rows

def at_least(row):
    bands = [row[band] for band in LEAD_BANDS]
    return [row['already'] - sum(bands[:index]) for index in range(len(bands) - 1)] + [bands[-1]]

def detector_of(label):
    return next((name for folder, name in FOLDER.items() if folder in label), label)

leads = {level: read_lead(level) for level in LEVELS}
leads = {level: rows for level, rows in leads.items() if rows}
if leads:
    figure, axes = plt.subplots(1, len(leads), figsize=(6.5 * len(leads), 4.2), constrained_layout=True,
                                squeeze=False)
    for axis, (level, rows) in zip(axes[0], leads.items()):
        labels = list(rows)
        for index, label in enumerate(labels):
            if not label.startswith('starts of'):
                continue
            starts_of = detector_of(label.split(',')[0])
            other = MODEL if starts_of == SIMPLE else SIMPLE
            color = 'tab:red' if other == MODEL else 'tab:blue'
            axis.plot(AT_LEAST, at_least(rows[label]), marker='o', color=color,
                      label=f'{other} già acceso a inizio episodio di {starts_of}')
            chance = labels[index + 1] if index + 1 < len(labels) and labels[index + 1].startswith('chance') else None
            if chance:
                axis.plot(AT_LEAST, at_least(rows[chance]), marker='.', linestyle='--', color=color, alpha=0.7,
                          label='  caso')
        axis.set_xticks(AT_LEAST, [f'≥{hours} h' for hours in AT_LEAST])
        axis.set(ylim=(0, None), xlabel='già acceso da almeno', ylabel='% degli inizi di episodio',
                 title=f'Segnalati al {level}%')
        axis.legend(fontsize=7)
    show(figure, 'b1_anticipo_tempo')
    for level, rows in leads.items():
        print(f'Segnalati al {level}%: tabella dello script')
        display(pd.DataFrame(rows).T.round(1))

## B2. Nello spazio, nelle celle intorno

All'inizio di ogni episodio di "normale + variazione" in una cella, il modello segnalava **le celle intorno**? Si
guardano anelli di celle a distanza crescente (in celle della griglia da 0,5°, contando il più grande fra lo
scarto di riga e di colonna) e diversi istanti prima dell'inizio. Tre "precursori":
- **modello**: il modello segnala almeno una cella dell'anello;
- **normale + variazione stesso**: la perturbazione che arriva dalle celle vicine, vista dagli score semplici. È il
  riferimento: un anticipo del modello conta se è più frequente di questo;
- **solo modello**: il modello segnala una cella dell'anello che "normale + variazione" in quel momento non
  segnala.

Colore: quante volte la quota supera il caso (la stessa misura in celle quiete che non cominciano nulla), in scala
logaritmica: rosso sopra il caso, blu sotto. Un anello lontano contiene più celle, quindi anche il suo caso è più
alto: per questo il colore è il rapporto. Nei riquadri, la quota degli inizi (%) e il rapporto. Le caselle con "—"
sono fissate dalla definizione di inizio (il rilevatore stesso nella sua cella) e non dicono nulla. La seconda riga
è il contrario: inizi del modello, precursori gli score semplici. Script `check_era5_spatial_lead.py`, tabella
`anticipo_spaziale_<livello>.csv`.

In [ ]:
RING_LABELS = {0: '0 (stessa cella)', 1: '1–2 (≈55–110 km)', 3: '3–5 (≈165–280 km)', 6: '6–10 (≈330–560 km)'}

def precursor_name(label, starts_of):
    if label.endswith('(itself)'):
        return f'{starts_of} stesso'
    if ' not ' in label:
        return f'solo {detector_of(label.split(" not ")[0])}'
    return detector_of(label)

for level in LEVELS:
    path = find(f'anticipo_spaziale_{level}.csv')
    if path is None:
        continue
    table = pd.read_csv(path)
    table['ratio'] = table['share'] / table['chance'].where(table['chance'] > 0)
    # Fixed by the definition of a start: the score itself at its cell, and what only the other flags there at 0 h.
    fixed = (table['ring_high'] == 0) & (table['precursor'].str.endswith('(itself)')
                                        | (table['precursor'].str.contains(' not ') & (table['lag_hours'] == 0)))
    table.loc[fixed, ['share', 'ratio']] = np.nan
    order = [folder for folder in ('simple_max', 'gen_clim') if folder in set(table['starts_of'])]
    precursors = {folder: list(dict.fromkeys(table.loc[table['starts_of'] == folder, 'precursor'])) for folder in order}
    columns = max(len(names) for names in precursors.values())
    norm = TwoSlopeNorm(vmin=-2, vcenter=0, vmax=4)  # log2 del rapporto: da ×¼ a ×16
    figure, axes = plt.subplots(len(order), columns, figsize=(5.2 * columns, 3.6 * len(order)),
                                constrained_layout=True, squeeze=False)
    for row, folder in enumerate(order):
        starts_of = detector_of(folder)
        for column, label in enumerate(precursors[folder]):
            part = table[(table['starts_of'] == folder) & (table['precursor'] == label)]
            grid_share = part.pivot(index='ring_low', columns='lag_hours', values='share')
            grid_ratio = part.pivot(index='ring_low', columns='lag_hours', values='ratio')
            axis = axes[row, column]
            with np.errstate(divide='ignore'):
                shade = np.clip(np.log2(grid_ratio.to_numpy(dtype=float)), -2, 4)
            image = axis.imshow(shade, cmap='RdBu_r', norm=norm, aspect='auto')
            for i in range(grid_ratio.shape[0]):
                for j in range(grid_ratio.shape[1]):
                    ratio, share = grid_ratio.iat[i, j], grid_share.iat[i, j]
                    text = ('—' if not np.isfinite(share) else f'{share:.0f}%'
                            + ('' if not np.isfinite(ratio) else f'\n×{ratio:.1f}'))
                    axis.text(j, i, text, ha='center', va='center', fontsize=7)
            axis.set_xticks(range(grid_ratio.shape[1]), [f'{lag} h' for lag in grid_ratio.columns])
            axis.set_yticks(range(grid_ratio.shape[0]), [RING_LABELS.get(ring, ring) for ring in grid_ratio.index],
                            fontsize=7)
            axis.set(xlabel="ore prima dell'inizio", title=f'inizi di {starts_of}\nprecursore: '
                                                           f'{precursor_name(label, starts_of)}')
            axis.grid(False)
        for column in range(len(precursors[folder]), columns):
            axes[row, column].axis('off')
    bar = figure.colorbar(image, ax=axes, shrink=0.6, label='volte il caso')
    bar.set_ticks([-2, -1, 0, 1, 2, 3, 4], labels=['×¼', '×½', '×1', '×2', '×4', '×8', '≥×16'])
    figure.suptitle(f'Segnalati al {level}%: chi segnala le celle intorno prima che un episodio cominci')
    show(figure, f'b2_anticipo_spazio_{level}')
    counts = table.groupby('starts_of')['starts'].first()
    print(f'Segnalati al {level}%: ' + ', '.join(f'{int(counts[folder])} inizi di episodio di {detector_of(folder)}'
                                                 for folder in order)
          + '. Il rilevatore stesso nella stessa cella vale 100% a 0 h e 0% prima, per definizione di inizio.')

---

## Come leggere le figure

- **A1** dice quanto il modello aggiunge, **A2** se quello che aggiunge è diverso da ciò che vedono gli altri o
  solo appena sotto la loro soglia, **A3** se è tempo davvero insolito: per il gruppo "solo modello" conta stare
  sopra 1, e il confronto con "nessuno" e con "solo semplici".
- **B1** è l'anticipo nella stessa cella: conta la distanza fra la curva e il suo caso, e il confronto fra le due
  direzioni.
- **B2** è l'anticipo nello spazio: conta che il modello stia sopra il caso negli anelli intorno e nelle ore prima,
  più di quanto ci stiano gli score semplici stessi (la perturbazione che arriva). Il precursore "solo modello"
  dice quanto di quell'anticipo gli score semplici non vedono affatto.
- L'anticipo è rispetto agli altri rilevatori, non una previsione: ERA5 arriva con qualche giorno di ritardo.
- Run `_ep3`, un solo seed, climatologia e soglie stimate sugli anni di test.